# 手写数字识别实训项目
# （深度学习入门-手写识别）
# 邯郸职业技术学院大数据综合实训深度学习入门项目

基于百度 AI Studio 经典 MNIST 手写识别项目改造，面向**邯郸职业技术学院大数据技术专业深度学习实训课程**：

- ✅ 参数可一键调整（学习率 / 轮数 / 网络结构），课堂演示参数效果
- ✅ 数据与日志自动留档（数据集记录 + 训练日志 + 曲线）
- ✅ 自带输入图片示例与识别结果
- ✅ 训练过程有损失/准确率曲线可视化
- ✅ 代码按功能拆成多个 `.py`，本 notebook 仅用于**讲解与演示**

> ⚠️ 实际训练/推理请在终端运行 `.py` 文件：
> `python main.py --mode train ...` 与 `python main.py --mode inference ...`
> 本 notebook 里也用 `!python ...` 直接调用它们。


## 0. 环境准备

AI Studio（CPU 环境）已预装 `paddlepaddle`，无需重装。
如需在本地运行，先装依赖：

```bash
pip install -r requirements.txt
```


In [ ]:
import paddle
print("PaddlePaddle 版本：", paddle.__version__)


## 1. 可调参数模块 `config.py`

所有"能调着玩"的超参数都集中在这里。课堂上改命令行参数即可看效果：

| 参数 | 含义 | 试试看 |
|---|---|---|
| `--lr` | 学习率 | 0.01（快但抖）vs 0.0005（稳但慢） |
| `--epochs` | 训练轮数 | 对比 2 轮 vs 10 轮 |
| `--model_type` | 网络结构 | `mlp` vs `lenet` vs `cnn` |
| `--optimizer` | 优化器 | `sgd` vs `adam` |
| `--batch_size` | 批大小 | 64 vs 256 |


In [ ]:
from config import Config
cfg = Config()
print("默认参数：")
for k, v in cfg.to_dict().items():
    print(f"  {k:14s}= {v}")


## 2. 数据加载与记录 `data_utils.py`

加载 MNIST（6万训练 / 1万测试，28×28 灰度），并生成**数据集加载记录** json 留档。


In [ ]:
from data_utils import load_dataset, save_load_record
train_loader, test_loader, rec = load_dataset(cfg)
print("数据集加载记录：")
print(rec)
save_load_record(rec, cfg.log_dir, cfg.run_name)


## 3. 三种网络结构 `model.py`

切换 `model_type` 即可对比不同结构：
- `mlp`：全连接，最基础
- `lenet`：经典卷积网络
- `cnn`：稍深卷积，准确率更高


In [ ]:
from model import build_model
model = build_model(cfg)
print(model)


## 4. 训练 + 日志 + 曲线 `train.py`

下面直接在终端跑 `.py`（真正干活的文件）。观察日志里的 loss/acc，
训练结束后会在 `logs/<run_name>/` 生成 `curves.png` 和 `history.json`。


In [ ]:
# 训练 3 轮做演示（CPU 几分钟即可）
!python main.py --mode train --model_type lenet --lr 0.001 --epochs 3 --run_name demo_run


## 5. 推理：输入图片示例 + 识别结果 `inference.py`

加载模型，挑 9 张测试图，保存每张输入图与综合结果拼图到 `outputs/examples/`。


In [ ]:
!python main.py --mode inference --run_name demo_run --num_show 9


## 6. 可视化结果

左：训练损失/准确率曲线；右：9 张示例（绿=预测正确，红=预测错误）。


In [ ]:
from IPython.display import Image, display
display(Image(filename="logs/demo_run/curves.png"))
display(Image(filename="outputs/examples/predictions.png"))


## 7. 课堂演示参数效果的几种玩法

1. **学习率对比**：`--lr 0.01` 看曲线剧烈震荡；`--lr 0.0005` 看收敛变慢。
2. **结构对比**：`--model_type mlp` 准确率明显低于 `lenet`/`cnn`，引出"为什么需要卷积"。
3. **轮数对比**：`--epochs 2` vs `--epochs 15`，观察是否过拟合（val_acc 掉、train_acc 高）。
4. **优化器对比**：`--optimizer sgd` vs `adam`，看收敛速度差异。

每次换个 `--run_name`，日志互不覆盖，方便课后复盘对比。


In [ ]:
# 示例：用 MLP + 大学习率 跑一轮，制造"对比组"
!python main.py --mode train --model_type mlp --lr 0.01 --epochs 3 --run_name compare_mlp
